# RiskBands Premium Benchmark

Este notebook mostra, com dados sint?ticos de cr?dito por safra, por que um binning que parece forte no agregado pode ficar fr?gil quando olhamos estabilidade temporal, cobertura, revers?es e volatilidade de comportamento entre vintages.

A compara??o usa tr?s lentes:

- `OptimalBinning` puro como baseline externa
- `RiskBands` est?tico como baseline interna sem foco temporal
- `RiskBands` selecionado por uma r?gua balanceada e cr?dito-orientada

O objetivo aqui n?o ? propaganda. Em alguns cen?rios o candidato est?tico continua sendo a melhor escolha. Em outros, o RiskBands troca para uma alternativa menos agressiva em IV bruto, por?m mais defend?vel no tempo.

In [1]:
from IPython.display import Markdown, display
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

import pandas as pd

ROOT = Path(r'D:\0_CienciaDados\1_Frameworks\RiskBands')
MODULE_PATH = ROOT / 'examples' / 'pd_vintage_benchmark' / 'pd_vintage_benchmark.py'
spec = spec_from_file_location('riskbands_benchmark_demo', MODULE_PATH)
module = module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = module
spec.loader.exec_module(module)

BENCHMARK_OBJECTIVE_KWARGS = module.BENCHMARK_OBJECTIVE_KWARGS
build_benchmark_visuals = module.build_benchmark_visuals
run_benchmark_suite = module.run_benchmark_suite

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

suite = run_benchmark_suite(samples_per_period=180)
suite['scenario_summary']

,scenario,scenario_title,best_temporal_candidate,best_balanced_candidate,selected_candidate,selected_equals_static,external_iv,static_iv,selected_iv,external_temporal_score,static_temporal_score,selected_temporal_score,external_objective_score,static_objective_score,selected_objective_score,selected_advantage_vs_static,selected_advantage_vs_external,selected_ranking_reversals,selected_basis,selected_alert_flags
0,composition_shift,Composition Shift,riskbands_temporal_quantile,riskbands_static,riskbands_static,True,1.596312,1.596312,1.596312,-1.075364,0.0,0.000000,-2.602180,0.040856,0.040856,0.000000,2.643036,0,discrimination-led,low_coverage
1,stable_credit,Stable Credit,riskbands_static_compact,riskbands_static,riskbands_static,True,1.594915,1.594915,1.594915,-0.167585,0.0,0.000000,-1.121758,0.050466,0.050466,0.000000,1.172224,0,discrimination-led,low_coverage
2,temporal_reversal,Temporal Reversal,riskbands_temporal_quantile,riskbands_temporal_quantile,riskbands_temporal_quantile,False,1.677150,1.677150,1.130749,-0.775519,0.0,0.316535,-2.084651,0.056993,0.306232,0.249239,2.390883,0,balanced,event_rate_volatility;woe_volatility;share_instability


## Como ler este benchmark

- `Stable Credit`: cen?rio em que a auditoria temporal n?o encontra motivo forte para abandonar o candidato mais discriminante.
- `Temporal Reversal`: cen?rio em que a leitura agregada engana e a vers?o balanceada do RiskBands troca para um candidato mais robusto.
- `Composition Shift`: cen?rio em que a mudan?a de composi??o piora a baseline externa, mas o trade-off ainda pode manter a escolha est?tica interna.

A r?gua balanceada deste benchmark usa pesos expl?citos mais alinhados a cr?dito, dando mais import?ncia a estabilidade temporal e mais puni??o para cobertura fraca, volatilidade e shortfall temporal.

In [2]:
display(Markdown('### Objective config usado no benchmark'))
pd.Series(BENCHMARK_OBJECTIVE_KWARGS, dtype='object')

### Objective config usado no benchmark

base_weights                                                          {'separability': 0.2, 'iv': 0.2, 'ks': 0.05, 'temporal_score': 0.55}
penalty_weights    {'coverage_gap': 0.35, 'event_rate_volatility': 0.35, 'woe_volatility': 0.12, 'share_volatility': 0.25, 'ranking_rev...
minimums                                                                        {'iv': 0.02, 'temporal_score': 0.1, 'coverage_ratio': 0.8}
dtype: object

In [3]:
def display_scenario(result):
    spec = result['scenario_spec']
    winner = result['winner_summary'][[
        'best_static_candidate',
        'best_temporal_candidate',
        'best_balanced_candidate',
        'selected_candidate',
        'winner_margin',
        'winner_rationale',
    ]]
    board = result['approach_board'][[
        'approach_label',
        'candidate_name',
        'iv',
        'ks',
        'temporal_score',
        'objective_score',
        'total_penalty',
        'coverage_ratio_min',
        'rare_bin_count',
        'ranking_reversal_period_count',
        'alert_flags',
    ]]
    profiles = result['candidate_profiles'][[
        'candidate_name',
        'candidate_role',
        'static_profile_score',
        'temporal_profile_score',
        'balanced_profile_score',
        'static_rank',
        'temporal_rank',
        'balanced_rank',
    ]].sort_values(['balanced_rank', 'temporal_rank', 'static_rank'])

    display(Markdown(f'## {spec.title}'))
    display(Markdown(spec.description))
    display(Markdown('### Approach board'))
    display(board)
    display(Markdown('### Profile winners'))
    display(winner)
    display(Markdown('### Candidate profile summary'))
    display(profiles)
    display(Markdown('### Key takeaways'))
    for takeaway in result['credit_takeaways']:
        display(Markdown(f'- {takeaway}'))

    figures = build_benchmark_visuals(result)
    display(figures['benchmark_board'])
    display(figures['metric_comparison'])
    display(figures['event_rate_curves'])
    display(figures['selected_heatmap'])
    display(figures['aggregate_vs_vintage'])
    display(figures['penalty_breakdown'])
    display(figures['score_distribution'])
    display(figures['sampling_preview'])

In [4]:
for scenario_name in ['stable_credit', 'temporal_reversal', 'composition_shift']:
    display_scenario(suite['scenarios'][scenario_name])

## Stable Credit

Mild drift and mostly monotonic ordering. The static solution can stay competitive and RiskBands may simply validate that choice.

### Approach board

,approach_label,candidate_name,iv,ks,temporal_score,objective_score,total_penalty,coverage_ratio_min,rare_bin_count,ranking_reversal_period_count,alert_flags
0,OptimalBinning puro,optimal_binning_pure,1.594915,0.5,-0.167585,-1.121758,1.532224,1.0,4,1,rare_bins;event_rate_volatility;woe_volatility;monotonic_break;ranking_reversal
1,RiskBands estatico,riskbands_static,1.594915,0.5,0.000000,0.050466,0.360000,0.0,0,0,low_coverage
2,RiskBands selecionado,riskbands_static,1.594915,0.5,0.000000,0.050466,0.360000,0.0,0,0,low_coverage


### Profile winners

...

- Cenario `composition_shift`: o candidato final do RiskBands foi `riskbands_static`.

- Neste caso o RiskBands concordou com a leitura estatica: a auditoria temporal nao encontrou ganho suficiente para justificar abandonar o candidato mais discriminante.

- O lider puramente temporal foi `riskbands_temporal_quantile`, mas o score balanceado final permaneceu com `riskbands_static` por um trade-off mais favoravel entre estabilidade e discriminacao.

- A baseline externa de OptimalBinning manteve IV competitivo, mas perdeu em score temporal, sugerindo fragilidade relevante para uso em credito.

- Racional do vencedor: Venceu mais pela discriminacao estatica do que pela estabilidade temporal. Drivers principais: iv_component=0.319; separability_component=0.065; ks_component=0.017. Penalizacoes relevantes: coverage_gap_penalty=0.280; temporal_shortfall_penalty=0.080. Alertas remanescentes: low_coverage.

## Leituras esperadas

### 1. O agregado pode esconder fragilidade
No cenário `Temporal Reversal`, a baseline externa e o baseline est?tico interno preservam IV forte, mas a leitura por vintage mostra perda de ordenação e pior estabilidade temporal. Esse ? o tipo de caso em que a regra `maximizar IV` conta uma hist?ria incompleta.

### 2. Nem sempre o temporal vence
No cenário `Stable Credit`, a troca para um candidato mais temporal n?o ? necess?ria. Isso é importante porque evita a impressão errada de que RiskBands sempre precisa sacrificar discrimina??o.

### 3. Mudança de composi??o n?o implica troca autom?tica
No cenário `Composition Shift`, a baseline externa sofre mais, mas o trade-off final ainda pode manter o candidato estático interno. A decisão correta depende da combinação entre discriminação, cobertura, volatilidade e revers?es.

## Limitações desta demo

- Os cenários são sintéticos e foram desenhados para expor trade-offs metodológicos, não para substituir validação em base real.
- A régua balanceada usada aqui é explícita e auditável, mas continua sendo uma escolha de benchmark; outras carteiras podem pedir pesos e pisos diferentes.
- O `TargetSampler` entra só como preview opcional para mostrar como prevalência por safra pode distorcer a leitura agregada. Ele não é o tema central do benchmark.

## Próximo passo natural

Se este benchmark fizer sentido para a sua carteira, o passo seguinte é repetir a mesma estrutura com uma variável real de bureau ou scorecard, mantendo as mesmas tabelas de auditoria e a mesma leitura por vintage.